We've used some datasets with PyTorch before.

But how do you get your own data into PyTorch?

One of the ways to do so is via: custom datasets.

### Domain libraries

Depending on what you're working on, vision, text, audio, recommendation, you'll want to look into each of the PyTorch domain libraries for existing data loading functions and customizable data loading functions.

In [ ]:
import torch
from torch import nn
import requests 
import os
import zipfile
from pathlib import Path

## 1. Get data

Our dataset is a subset of the Food101 dataset.

Food101 starts 101 different classes of food and 1000 images per class (750 training, 250 testing).

Our dataset starts with 3 classes of food and only 10% of the images (~75 training, 25 testing).

Why do this?

When starting out ML projects, it's important to try things on a small scale and then increase the scale when necessary.

The whole point is to speed up how fast you can experiment.

In [ ]:
os.makedirs("data", exist_ok=True)
image_path = Path("data/pizza_steak_sushi")

# If the image folder doesn't exist, download it and prepare it...
if os.path.isdir(image_path):
    print(f"{image_path} is already exists...skipping download")
else:
    os.makedirs(image_path, exist_ok=True)
    print(f"{image_path} is not exists, creating new one...")

In [ ]:
# Download pizza, steak and suhsi data
with open("data/pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    f.write(request.content)
    print("Downloading pizza, steak and sushi data...")

# Unzip pizza, steak, sushi data
with zipfile.ZipFile("data/pizza_steak_sushi.zip", "r") as zip_ref:
    zip_ref.extractall(image_path)
    print("Unzipping pizza, steak and sushi data...")

In [ ]:
def walk_through_dir(dir_path):
    for dir_path, dirnames, filenames in os.walk(dir_path):
        print(f"There are {len(dirnames)} directory and {len(filenames)} images in '{dir_path}'")

walk_through_dir(image_path)

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir, test_dir

### 2.1 Visualizing and image

Let's write some code to:

- Get all of the image paths
- Pick a random image path using Python's `random.choice()`
- Get the image class name using `pathlib.Path.parent.stem`
- Since we're working with images, let's open the image with Python's `PIL`
- We'll then show the image and print metadata

In [ ]:
from PIL import Image
import random

# 1. Get all image paths 
image_path_list = list(image_path.glob("*/*/*.jpg"))

# 2. Pick a random image path
random_image_path = random.choice(image_path_list)

# 3. Get image class from path name (the image class is the name of the directory where the image is stored)
image_class = random_image_path.parent.stem   #Ex: parent= data/pizza_steak_sushi/train/pizza , stem= pizza

# 4. Open image
img = Image.open(random_image_path)

# 5. Print metadata 
print(f"Image Path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

image_as_array = np.asarray(img)     # Turn the image into an array

plt.imshow(image_as_array)           # Plot the image with matplotlib
plt.axis(False)
print(f"Image Class: {image_class} | Image shape: {image_as_array.shape} -> [height, width, color channels] (HWC)");

## 3. Transforming data

Before we can use our image data with PyTorch:

Turn your target data into tensors (in our case, numerical representation of our images).               
Turn it into a `torch.utils.data.Dataset` and subsequently a `torch.utils.data.DataLoader`, we'll call these `Dataset` and `DataLoader`.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### 3.1 Transforming data with torchvision.transforms



In [ ]:
# Write a transform for image
data_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),          # Resize our images to 64x64
    transforms.RandomHorizontalFlip(p=0.5),   # Flip the images randomly on the horizontal
    transforms.ToTensor()                     # Turn the image into a torch.Tensor
])

data_transform(img).shape

Because `np.asarray(img)` follows the common image format used by NumPy and matplotlib: **HWC (height, width, channels)**.

But `transforms.ToTensor()` converts the image into PyTorch format, which is **CHW (channels, height, width)**.

So:
- NumPy / matplotlib → HWC
- PyTorch tensor → CHW

**ToTensor() changes both:**
1. the data type to a tensor
2.	the dimension order from HWC to CHW

In [ ]:
def plot_transformed_images(image_paths: list, transform, n=3, seed=None):
    """
    Selects random images from a path of images and loads/transforms 
    them then plots the original vs the transformed version.
    """
    if seed:
        random.seed(seed)

    random_image_paths = random.sample(image_paths, k=n)
    for image_path in random_image_paths:
        with Image.open(image_path) as f:
            fig, ax = plt.subplots(nrows = 1, ncols = 2)
            ax[0].imshow(f)
            ax[0].set_title(f"Original\nSize: {f.size}")
            ax[0].axis(False)

            # Transform and plot target image
            transformed_image = transform(f).permute(1,2,0)   # note we will need to change shape for matplotlib (C, H, W) -> (H, W, C)
            ax[1].imshow(transformed_image)
            ax[1].set_title(f"Transformed\nShape: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16, c="g")

plot_transformed_images(image_paths = image_path_list,
                       transform = data_transform,
                       n=3,
                       seed=42)

## 4. Option 1: Loading image data using ImageFolder

In [ ]:
# Use ImageFolder to create dataset(s)
from torchvision import datasets

#images -> tensor
train_data = datasets.ImageFolder(root = train_dir,
                                 transform = data_transform,     # a transform for the data
                                 target_transform = None)   #default=None   # a transform for the label/target

test_data = datasets.ImageFolder(root = test_dir,
                                transform = data_transform)

train_data, test_data

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
len(train_data), len(test_data)

In [ ]:
train_data[0]

In [ ]:
train_data.samples[0]

In [ ]:
random_idx = torch.randint(0, len(train_data), size=[1]).item()
image, label = train_data[random_idx]
plt.imshow(image.permute(1,2,0))
plt.title(class_names[label])
plt.axis(False);

### 4.1 Turn loaded images into DataLoader's

A `DataLoader` is going to help us turn our `Dataset`'s into iterables and we can customise the `batch_size` so our model can see `batch_size` images at a time.

In [ ]:
BATCH_SIZE = 1
train_dataloader = DataLoader(dataset = train_data,
                             batch_size = BATCH_SIZE,
                             num_workers = os.cpu_count(),   
                             shuffle = True)                

test_dataloader = DataLoader(dataset = test_data,
                            batch_size = BATCH_SIZE,
                            num_workers = os.cpu_count(),
                            shuffle = True)

#num_workers sets how many subprocesses are used to load the data.
#More workers can make data loading faster because batches are prepared in parallel.
len(train_dataloader), len(test_dataloader)

In [ ]:
os.cpu_count()